# Subsample phospho data for learning curve experiment

In [ ]:
import random
import os 

ordered_ds_names = """
Primary-AML BoneMarrow SW480 MCF7
Daudi A673 Lung Kit255
U2OS Mono-Mac-1 MV-4-11 SKM-1
Primary-Gastro WM239A 143B.TK BT474
HeLa Primary-Ovarian Jurkat HCT116
HEK293 A431 DG75 M019i
A549 Fibroblast U937 Primary-Pancreas
HaCaT HUVEC H1975 Metastasis-Pancreas
OVAS KOC-7C HCCLM6 OVISE
Primary-Prostate MCF10A Primary-Melanoma SH-SY5Y
TOV-21-Primary Primary-BOEC Primary-Colorectal Platelets
LNCaP MDA-MB-231 H3255 11-18
HEPG2 Primary-Breast-MicroAndExo H1299 SK-N-BE
Colon Primary-Glioblastoma RKO Brain
Kasumi-1 Muscle COLO-205 P31.Fuj
HT-29 SW1398 THP1 ECV-304
ES2-Primary DLD-1 CACO-2 GB2
OVSAYO H358 Primary-Glioma Primary-Liver
HDMVEC DoHH2 SU-DHL-6 RL
PANC-05-04 CTS PC9 Capan-1
SU.86.86 HPAF-II BxPC-3 PANC-04-03
CFPAC-1 PANC-08-13 Capan-2 SW1990
RPMI-8226 Hs-700-T PANC-10-05 PL45
HPAC Hs-766-T PANC-03-27 OMP2
PANC-02-03 AsPC-1 CL1-0 U266B1
"""

splits = {'a':[],'b':[],'c':[],'d':[]}
for quart in ordered_ds_names.split("\n")[1:-1]:
    for ds,split_name in zip(quart.split(" "), ['a','b','c','d']):
        splits[split_name].append(ds)
#Add manually
splits['d'].append("ACHN")


train_dataset_names = splits['c'] + splits['d']
valid_dataset_names = splits['b']
test_dataset_names = splits ['a']

global_total = 0

def downsample_file(f, basename, factor, ds):
    global global_total
    local_total = 0
    outfile = 'data/'+factor+'/'+ds+'/'+basename+'.mgf'
    with open(f) as f_in:
        with open(outfile, 'w') as f_out:
            copying = True
            for line in f_in:
                if 'BEGIN' in line:
                    copying = random.random() < 0.5 or local_total == 0
                    if True:
                        global_total += 1
                        local_total += 1
                if copying:
                    f_out.write(line)
    print(local_total)

last_factor = '1'
for factor in ['2', '4', '8', '16', '32', '64', '128', '256', '512', '1024']:
    for sp in ['a','b','c','d']:
        print(f"Split {sp}")
        for ds in splits[sp]:
            print(ds)
            if not os.path.exists('data/'+factor):
                os.makedirs('data/'+factor)
            if not os.path.exists('data/'+factor+'/'+ds):
                os.makedirs('data/'+factor+'/'+ds)
            downsample_file(f'data/{last_factor}/{ds}/phospho.mgf', 'phospho', factor, ds)
            downsample_file(f'data/{last_factor}/{ds}/non_phospho.mgf', 'non_phospho', factor, ds)
            last_factor = factor
            print()

print(global_total)

### Combine embeddings

In [ ]:
#Write extended embeddings to disk
import pickle
import torch
from tqdm import tqdm
import numpy as np

#Copied from Supp Table 2 in AHLF paper. Missing ACHN in d split.
ordered_ds_names = """
HeLa Primary-Ovarian Jurkat HCT116
HEK293 A431 DG75 M019i
Primary-AML BoneMarrow SW480 MCF7
Daudi A673 Lung Kit255
U2OS Mono-Mac-1 MV-4-11 SKM-1
Primary-Gastro WM239A 143B.TK BT474
A549 Fibroblast U937 Primary-Pancreas
HaCaT HUVEC H1975 Metastasis-Pancreas
OVAS KOC-7C HCCLM6 OVISE
Primary-Prostate MCF10A Primary-Melanoma SH-SY5Y
TOV-21-Primary Primary-BOEC Primary-Colorectal Platelets
LNCaP MDA-MB-231 H3255 11-18
HEPG2 Primary-Breast-MicroAndExo H1299 SK-N-BE
Colon Primary-Glioblastoma RKO Brain
Kasumi-1 Muscle COLO-205 P31.Fuj
HT-29 SW1398 THP1 ECV-304
ES2-Primary DLD-1 CACO-2 GB2
OVSAYO H358 Primary-Glioma Primary-Liver
HDMVEC DoHH2 SU-DHL-6 RL
PANC-05-04 CTS PC9 Capan-1
SU.86.86 HPAF-II BxPC-3 PANC-04-03
CFPAC-1 PANC-08-13 Capan-2 SW1990
RPMI-8226 Hs-700-T PANC-10-05 PL45
HPAC Hs-766-T PANC-03-27 OMP2
PANC-02-03 AsPC-1 CL1-0 U266B1
"""

for downsample in ['1', '2', '4', '8', '16', '32', '64', '128', '256', '512', '1024']:
    print('Downsample')
    splits = {'a':[],'b':[],'c':[],'d':[]}
    for quart in ordered_ds_names.split("\n")[1:-1]:
        for ds,split_name in zip(quart.split(" "), ['a','b','c','d']):
            splits[split_name].append(ds)
    #Add manually
    splits['d'].append("ACHN")

    def parse_mgf(file_path):
        spectra = []
        with open(file_path, 'r') as file:
            spectrum = {}
            for line in file:
                line = line.strip()
                if line.startswith("BEGIN IONS"):
                    spectrum = {}
                elif line.startswith("END IONS"):
                    spectra.append(spectrum)
                elif "=" in line:
                    key, value = line.split("=", 1)
                    if key.lower() == "pepmass":
                        spectrum["m/z"] = float(value.split()[0])  # Some files may include intensity
                    elif key.lower() == "charge":
                        spectrum["charge"] = int(value.replace("+", "").replace("-", ""))
            return spectra

    prec_charge = {}
    for sp in ['a','b','c','d']:
        print(sp)
        for ds in tqdm(splits[sp]):
            phos_prec_charge = parse_mgf(f'data/{downsample}/{ds}/phospho.mgf')
            non_phos_prec_charge= parse_mgf(f'data/{downsample}/{ds}/non_phospho.mgf')
            prec_charge[ds] = phos_prec_charge + non_phos_prec_charge
    print()
    print("Parsed precursors")
    embeds = {}
    phospho_labels = {}
    for sp in ['a','b','c','d']:
        print(sp)
        for ds in tqdm(splits[sp]):
            phos_embeds = [emb.numpy() for emb in torch.load(f"embeddings/{downsample}/{ds}/phospho.pt")]
            non_phos_embeds = [emb.numpy() for emb in torch.load(f"embeddings/{downsample}/{ds}/non-phospho.pt")]
            phospho_labels[ds] = [1] * len(phos_embeds) + [0] * len(non_phos_embeds)
            embeds[ds] = phos_embeds + non_phos_embeds

    for ds in tqdm(embeds.keys()):
        embeds[ds] = [np.append(emb, [prec['m/z'], prec['charge']]) for emb, prec in zip(embeds[ds], prec_charge[ds])]

    with open(f"embeddings/{downsample}/PXD012174_embeddings_w_precursor.pkl", "wb") as f:
        pickle.dump(embeds, f, protocol=pickle.HIGHEST_PROTOCOL)
    
    with open(f"embeddings/{downsample}/PXD012174_labels.pkl", "wb") as f:
        pickle.dump(phospho_labels, f, protocol=pickle.HIGHEST_PROTOCOL)
    

# Combine mgfs into single run

In [ ]:
import random
import os 

ordered_ds_names = """
Primary-AML BoneMarrow SW480 MCF7
Daudi A673 Lung Kit255
U2OS Mono-Mac-1 MV-4-11 SKM-1
Primary-Gastro WM239A 143B.TK BT474
HeLa Primary-Ovarian Jurkat HCT116
HEK293 A431 DG75 M019i
A549 Fibroblast U937 Primary-Pancreas
HaCaT HUVEC H1975 Metastasis-Pancreas
OVAS KOC-7C HCCLM6 OVISE
Primary-Prostate MCF10A Primary-Melanoma SH-SY5Y
TOV-21-Primary Primary-BOEC Primary-Colorectal Platelets
LNCaP MDA-MB-231 H3255 11-18
HEPG2 Primary-Breast-MicroAndExo H1299 SK-N-BE
Colon Primary-Glioblastoma RKO Brain
Kasumi-1 Muscle COLO-205 P31.Fuj
HT-29 SW1398 THP1 ECV-304
ES2-Primary DLD-1 CACO-2 GB2
OVSAYO H358 Primary-Glioma Primary-Liver
HDMVEC DoHH2 SU-DHL-6 RL
PANC-05-04 CTS PC9 Capan-1
SU.86.86 HPAF-II BxPC-3 PANC-04-03
CFPAC-1 PANC-08-13 Capan-2 SW1990
RPMI-8226 Hs-700-T PANC-10-05 PL45
HPAC Hs-766-T PANC-03-27 OMP2
PANC-02-03 AsPC-1 CL1-0 U266B1
"""

splits = {'a':[],'b':[],'c':[],'d':[]}
for quart in ordered_ds_names.split("\n")[1:-1]:
    for ds,split_name in zip(quart.split(" "), ['a','b','c','d']):
        splits[split_name].append(ds)
#Add manually
splits['d'].append("ACHN")

train_dataset_names = splits['c'] + splits['d']
valid_dataset_names = splits['b']
test_dataset_names = splits ['a']

global_total = 0

def add_scans_to_file(f, basename):
    global global_total
    local_total = 0
    outfile = 'data/test/'+basename+'_32.mgf'
    with open(f) as f_in:
        with open(outfile, 'a') as f_out:
            copying = True
            for line in f_in:
                if 'BEGIN' in line:
                    copying = True
                    global_total += 1
                    local_total += 1
                if copying:
                    f_out.write(line)
    print(local_total)

for sp in ['a']:
    print(f"Split {sp}")
    for ds in splits[sp]:
        add_scans_to_file(f'data/32/{ds}/phospho.mgf', 'phospho_test_combined_32')
        add_scans_to_file(f'data/32/{ds}/non_phospho.mgf', 'non_phospho_test_combined_32')

print(global_total)